# HumanForYou - Quels facteurs expliquent les départs ? Optimisation et recommandations

Le notebook précédent a montré que le Random Forest est le meilleur modèle. Mais un modèle qui prédit sans expliquer ne répond qu'à la moitié de la question posée par HumanForYou.

La direction veut savoir **pourquoi** ses employés partent, pas seulement **lesquels** vont partir. Ce notebook répond à cette question en deux temps :
1. On optimise le RF pour avoir les meilleures prédictions possibles
2. On analyse ce que le modèle nous dit sur les facteurs de départ, avec des chiffres concrets issus des données brutes

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import (classification_report, roc_auc_score,
                                      f1_score, ConfusionMatrixDisplay, roc_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RAW    = "../data/raw/"
PROC   = "../data/processed/"
MODELS = "../models/" 

## 1. Chargement des données pour la modélisation

In [ ]:
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(PROC + "dataset_encoded.csv")

X = df.drop(columns=["Attrition", "EmployeeID"])
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation : fit uniquement sur X_train pour eviter toute fuite
cols_norm = joblib.load(MODELS + "cols_norm.pkl")
cols_to_scale = [c for c in cols_norm if c in X_train.columns]

scaler = StandardScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Partants dans le test : {y_test.sum()} sur {len(y_test)} ({y_test.mean()*100:.1f}%)")


## 2. Point de départ : RF avec les paramètres par défaut

On réentraîne le RF de base pour avoir une référence claire avant optimisation.

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
)
rf_base.fit(X_train, y_train)

y_pred_base  = rf_base.predict(X_test)
y_proba_base = rf_base.predict_proba(X_test)[:, 1]

f1_base  = f1_score(y_test, y_pred_base)
auc_base = roc_auc_score(y_test, y_proba_base)

print(f"RF de base  ->  F1 : {f1_base:.4f} | AUC : {auc_base:.4f}")

## 3. Recherche des meilleurs hyperparamètres

Les hyperparamètres, c'est ce qu'on règle avant l'entraînement :
- `n_estimators` : combien d'arbres dans la forêt
- `max_depth` : jusqu'où chaque arbre peut pousser (None = pas de limite)

On teste 8 combinaisons et on évalue chacune par validation croisée à 5 folds. C'est plus transparent que de laisser une fonction automatique fouiller dans tous les sens.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = [
    {"n_estimators": 100, "max_depth": None},
    {"n_estimators": 100, "max_depth": 20},
    {"n_estimators": 200, "max_depth": None},
    {"n_estimators": 200, "max_depth": 20},
    {"n_estimators": 300, "max_depth": None},
    {"n_estimators": 300, "max_depth": 20},
    {"n_estimators": 300, "max_depth": 30},
    {"n_estimators": 500, "max_depth": 30},
]

resultats = []
print(f"  {'n_estimators':>15} {'max_depth':>12} {'F1 moyen':>12} {'Std':>8}")
print("  " + "-"*52)

for params in grid:
    rf = RandomForestClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1, **params
    )
    scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring="f1")
    resultats.append({**params, "f1_mean": scores.mean(), "f1_std": scores.std()})
    print(f"  {params['n_estimators']:>15} {str(params['max_depth']):>12} {scores.mean():>12.4f} {scores.std():>8.4f}")

resultats_df = pd.DataFrame(resultats)
best_idx = resultats_df["f1_mean"].idxmax()
best_params = resultats_df.loc[best_idx]
print(f"\nMeilleure config : n_estimators={int(best_params['n_estimators'])}, max_depth={best_params['max_depth']}")
print(f"F1 moyen en CV   : {best_params['f1_mean']:.4f} +/- {best_params['f1_std']:.4f}")

## 4. Modèle final optimisé

In [ ]:
max_d = None if pd.isna(best_params["max_depth"]) else int(best_params["max_depth"])

rf_opti = RandomForestClassifier(
    n_estimators=int(best_params["n_estimators"]),
    max_depth=max_d,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_opti.fit(X_train, y_train)

y_pred_opti  = rf_opti.predict(X_test)
y_proba_opti = rf_opti.predict_proba(X_test)[:, 1]

f1_opti  = f1_score(y_test, y_pred_opti)
auc_opti = roc_auc_score(y_test, y_proba_opti)

print("=== Comparaison avant / apres optimisation ===")
print(f"{'RF de base':<35} F1 = {f1_base:.4f} | AUC = {auc_base:.4f}")
print(f"{'RF optimise':<35} F1 = {f1_opti:.4f} | AUC = {auc_opti:.4f}")
print()
print("--- Rapport complet (modele optimise) ---")
print(classification_report(y_test, y_pred_opti, target_names=["Reste (0)", "Part (1)"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opti,
    display_labels=["Reste", "Part"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Matrice de confusion - RF optimise", fontweight="bold")

for label, y_proba, color in [
    ("RF de base",  y_proba_base, "steelblue"),
    ("RF optimise", y_proba_opti, "seagreen")
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})", color=color)

axes[1].plot([0, 1], [0, 1], "k--", label="Aleatoire")
axes[1].set_xlabel("Taux faux positifs")
axes[1].set_ylabel("Rappel")
axes[1].set_title("Courbes ROC - avant / apres optimisation", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../outputs/rf_optimise_evaluation.png", bbox_inches="tight")
plt.show()

## 4b. Optimisation du seuil de decision

Le modele optimise produit **0 faux positif mais beaucoup de faux negatifs** avec le seuil par defaut (0.5). Dans un contexte RH, les couts ne sont pas symetriques : rater un partant est bien plus couteux que signaler un employe qui finalement reste.

On peut abaisser le seuil de decision pour capturer plus de departs reels. Mais ce choix doit etre fait rigoureusement :

- **A ne pas faire** : tester les seuils sur le jeu de test et garder le meilleur. Le test set deviendrait un jeu de validation de fait, et les resultats finaux seraient optimistes.
- **Ce qu'on fait** : rechercher le meilleur seuil par **validation croisee sur X_train uniquement** (5 folds, memes parametres que le RF optimise). Le jeu de test n'est touche qu'une seule fois, a la fin, pour la mesure finale.


In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
import numpy as np

seuils = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
cv_seuil = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultats_seuil = []

print("Recherche du seuil optimal par validation croisee (5 folds sur X_train)")
print(f"{'Seuil':>7} {'F1 moy CV':>11} {'Std':>7} {'FN moy':>8} {'FP moy':>8}")
print("  " + "-"*46)

for s in seuils:
    f1_folds, fn_folds, fp_folds = [], [], []

    for train_idx, val_idx in cv_seuil.split(X_train, y_train):
        X_tr  = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr  = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        max_d = None if pd.isna(best_params["max_depth"]) else int(best_params["max_depth"])
        rf_fold = RandomForestClassifier(
            n_estimators=int(best_params["n_estimators"]),
            max_depth=max_d,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
        rf_fold.fit(X_tr, y_tr)
        proba_val  = rf_fold.predict_proba(X_val)[:, 1]
        y_pred_val = (proba_val >= s).astype(int)

        f1_folds.append(f1_score(y_val, y_pred_val, zero_division=0))
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred_val, labels=[0, 1]).ravel()
        fn_folds.append(fn)
        fp_folds.append(fp)

    resultats_seuil.append({
        "seuil"   : s,
        "f1_mean" : np.mean(f1_folds),
        "f1_std"  : np.std(f1_folds),
        "fn_mean" : np.mean(fn_folds),
        "fp_mean" : np.mean(fp_folds),
    })
    print(f"  {s:>5.2f} {np.mean(f1_folds):>11.4f} {np.std(f1_folds):>7.4f} "
          f"{np.mean(fn_folds):>8.1f} {np.mean(fp_folds):>8.1f}")


In [ ]:
# Seuil choisi : celui qui maximise le F1 moyen en validation croisee
res_df  = pd.DataFrame(resultats_seuil)
best_s  = res_df.loc[res_df["f1_mean"].idxmax(), "seuil"]
best_cv = res_df.loc[res_df["f1_mean"].idxmax(), "f1_mean"]
best_std= res_df.loc[res_df["f1_mean"].idxmax(), "f1_std"]

print(f"Seuil optimal (CV) : {best_s}  ->  F1 CV = {best_cv:.4f} +/- {best_std:.4f}")
print()

# Application sur le test set : une seule fois, mesure finale
y_pred_final = (y_proba_opti >= best_s).astype(int)

print("--- Rapport final sur le jeu de test (seuil CV-optimal) ---")
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_final, target_names=["Reste (0)", "Part (1)"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opti,
    display_labels=["Reste", "Part"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Seuil par defaut (0.50)", fontweight="bold")
axes[0].set_xlabel("Predit")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=["Reste", "Part"],
    cmap="Greens", ax=axes[1]
)
axes[1].set_title(f"Seuil CV-optimal ({best_s:.2f})", fontweight="bold")
axes[1].set_xlabel("Predit")

plt.suptitle("Impact du seuil de decision : avant / apres optimisation par CV",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/seuil_optimise_confusion.png", bbox_inches="tight")
plt.show()

f1_final  = f1_score(y_test, y_pred_final)
auc_final = auc_opti
print(f"F1 final (test, seuil {best_s:.2f}) : {f1_final:.4f}")
print(f"AUC final (test)                  : {auc_final:.4f}  [inchange, metrique probabiliste]")


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

seuils_vals = [r["seuil"]   for r in resultats_seuil]
f1_means    = [r["f1_mean"] for r in resultats_seuil]
f1_stds     = [r["f1_std"]  for r in resultats_seuil]
fn_means    = [r["fn_mean"] for r in resultats_seuil]
fp_means    = [r["fp_mean"] for r in resultats_seuil]

ax1.errorbar(seuils_vals, f1_means, yerr=f1_stds,
             fmt="o-", color="steelblue", capsize=5, label="F1 moyen CV (+/- std)")
ax2.plot(seuils_vals, fn_means, "x--", color="tomato",   label="FN moyen CV")
ax2.plot(seuils_vals, fp_means, "s--", color="darkorange", label="FP moyen CV")

ax1.axvline(best_s, color="gray", linestyle=":", label=f"Seuil choisi ({best_s:.2f})")
ax1.set_xlabel("Seuil de decision")
ax1.set_ylabel("F1 (validation croisee)")
ax2.set_ylabel("Nombre moyen de FN / FP par fold", color="tomato")
ax2.tick_params(axis="y", labelcolor="tomato")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.title("Choix du seuil par validation croisee sur X_train", fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/seuil_courbe_tradeoff.png", bbox_inches="tight")
plt.show()

print("Note : ces courbes sont calculees sur X_train (CV). Le test set n'a pas ete utilise pour ce choix.")


## 5. Quels facteurs expliquent les départs ?

C'est la question centrale du projet. Le Random Forest calcule nativement l'importance de chaque variable via la réduction moyenne de l'impureté de Gini : plus une variable est utilisée pour séparer efficacement les partants des restants dans les arbres, plus son score est élevé.

Ce graphique répond directement à la demande de HumanForYou : **sur quoi agir pour réduire le taux de rotation de 15 % ?**

In [ ]:
importances = pd.Series(rf_opti.feature_importances_, index=X.columns)
top20 = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("Importance (reduction Gini moyenne)")
plt.title("Facteurs les plus predictifs du depart - RF optimise", fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/feature_importance.png", bbox_inches="tight")
plt.show()

print("Top 10 des facteurs de depart :")
for rang, (feat, imp) in enumerate(top20.head(10).items(), 1):
    print(f"  {rang:2}. {feat:<35} {imp:.4f}")

### 5b. Importance par permutation (verification)

L'importance Gini peut surestimer les variables numeriques continues. L'importance par permutation est plus robuste : elle mesure la chute de performance quand on melange aleatoirement les valeurs d'une variable. Si le modele se degrade beaucoup, la variable est vraiment importante.


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf_opti, X_test, y_test,
    n_repeats=10, random_state=42, scoring="f1", n_jobs=-1
)

perm_series = pd.Series(perm.importances_mean, index=X.columns)
top20_perm  = perm_series.sort_values(ascending=False).head(20)

# Comparaison Gini vs Permutation (top 15 communs)
top15_gini = importances.sort_values(ascending=False).head(15).index.tolist()
top15_perm = top20_perm.head(15).index.tolist()
communs = [f for f in top15_perm if f in top15_gini]
print(f"Variables dans le top 15 des deux methodes ({len(communs)}/15) :")
for f in communs:
    print(f"  - {f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

importances.sort_values(ascending=False).head(15).sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue"
)
axes[0].set_title("Importance Gini (top 15)", fontweight="bold")
axes[0].set_xlabel("Reduction impurete moyenne")

top20_perm.head(15).sort_values().plot(
    kind="barh", ax=axes[1], color="seagreen"
)
axes[1].set_title("Importance Permutation (top 15)", fontweight="bold")
axes[1].set_xlabel("Chute de F1 moyenne")

plt.suptitle("Gini vs Permutation : convergence des deux methodes", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/importance_gini_vs_permutation.png", bbox_inches="tight")
plt.show()


## 6. Combien de différence concrètement ?

L'importance des variables nous dit *quelles* variables comptent. Maintenant, on regarde les valeurs brutes pour comprendre *dans quel sens* et *de combien*. On charge les fichiers source originaux (non normalisés) pour avoir des chiffres interprétables : roupies, années, jours.

In [ ]:
# Chargement des donnees brutes (non normalisees)
general = pd.read_csv(RAW + "general_data.csv")
survey  = pd.read_csv(RAW + "employee_survey_data.csv")
manager = pd.read_csv(RAW + "manager_survey_data.csv")

# Calcul des features de badgeage
in_time  = pd.read_csv(RAW + "in_time.csv",  index_col=0)
out_time = pd.read_csv(RAW + "out_time.csv", index_col=0)
in_dt    = in_time.apply(pd.to_datetime, errors="coerce")
out_dt   = out_time.apply(pd.to_datetime, errors="coerce")
work_hours = (out_dt - in_dt).apply(lambda col: col.dt.total_seconds() / 3600)

badge = pd.DataFrame({
    "EmployeeID"       : general["EmployeeID"].values,
    "avg_hours_per_day": work_hours.mean(axis=1).values,
    "days_absent"      : work_hours.isna().sum(axis=1).values,
    "days_over_8h"     : (work_hours > 8).sum(axis=1).values,
})

# Fusion des 4 sources sur EmployeeID
brut = general.merge(survey,  on="EmployeeID", how="left")
brut = brut.merge(manager, on="EmployeeID", how="left")
brut = brut.merge(badge,   on="EmployeeID", how="left")

# Imputation mediane (coherent avec le preprocessing)
for col in brut.select_dtypes(include="number").columns:
    brut[col] = brut[col].fillna(brut[col].median())

restants = brut[brut["Attrition"] == "No"]
partants = brut[brut["Attrition"] == "Yes"]

print(f"Restants : {len(restants)} | Partants : {len(partants)}")

In [ ]:
vars_comparaison = {
    "MonthlyIncome"          : "Salaire mensuel (roupies)",
    "YearsAtCompany"         : "Anciennete (annees)",
    "YearsSinceLastPromotion": "Annees sans promotion",
    "TotalWorkingYears"      : "Annees experience totale",
    "DistanceFromHome"       : "Distance domicile-travail (km)",
    "TrainingTimesLastYear"  : "Formations en 2015 (jours)",
    "avg_hours_per_day"      : "Heures travaillees / jour",
    "days_absent"            : "Jours d'absence en 2015",
    "days_over_8h"           : "Jours depasses 8h",
    "EnvironmentSatisfaction": "Satisfaction environnement (1-4)",
    "JobSatisfaction"        : "Satisfaction au travail (1-4)",
    "WorkLifeBalance"        : "Equilibre vie pro/perso (1-4)",
    "JobInvolvement"         : "Implication au travail (1-4)",
    "StockOptionLevel"       : "Investissement actions (0-3)",
}

lignes = []
for col, label in vars_comparaison.items():
    if col not in brut.columns:
        continue
    m_r = restants[col].mean()
    m_p = partants[col].mean()
    delta = m_p - m_r
    delta_pct = (delta / m_r * 100) if m_r != 0 else 0
    lignes.append({
        "Variable"     : label,
        "Restants (moy)": round(m_r, 1),
        "Partants (moy)": round(m_p, 1),
        "Difference"   : round(delta, 1),
        "Variation"    : f"{delta_pct:+.1f}%"
    })

compar = pd.DataFrame(lignes).set_index("Variable")
print(compar.to_string())

In [ ]:
# Visualisation des 6 variables les plus discriminantes (valeurs brutes)
vars_plot = [
    ("MonthlyIncome",           "Salaire mensuel (roupies)"),
    ("YearsAtCompany",          "Anciennete (annees)"),
    ("YearsSinceLastPromotion", "Annees sans promotion"),
    ("JobSatisfaction",         "Satisfaction travail (1-4)"),
    ("EnvironmentSatisfaction", "Satisfaction environnement (1-4)"),
    ("days_absent",             "Jours d'absence 2015"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (col, label) in enumerate(vars_plot):
    data_r = restants[col]
    data_p = partants[col]
    axes[i].hist(data_r, bins=20, alpha=0.6, label="Reste", color="steelblue", density=True)
    axes[i].hist(data_p, bins=20, alpha=0.6, label="Part",  color="tomato",    density=True)
    axes[i].set_title(label, fontweight="bold")
    axes[i].legend()

plt.suptitle("Distribution des variables cles : qui reste vs qui part (valeurs brutes)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/profil_partants_vs_restants.png", bbox_inches="tight")
plt.show()

## 7. Recommandations pour HumanForYou

Chaque recommandation repose sur un ecart mesure et significatif dans les donnees, confirme par l'importance calculee par le modele. Les variables dont l'ecart est inferieur a 5% (distance domicile-travail, absenteisme) ont ete ecartees car non actionnables avec confiance.

In [ ]:
inc_r   = restants["MonthlyIncome"].mean()
inc_p   = partants["MonthlyIncome"].mean()
years_r = restants["YearsAtCompany"].mean()
years_p = partants["YearsAtCompany"].mean()
over8_r = restants["days_over_8h"].mean()
over8_p = partants["days_over_8h"].mean()
sat_r   = restants["JobSatisfaction"].mean()
sat_p   = partants["JobSatisfaction"].mean()
env_r   = restants["EnvironmentSatisfaction"].mean()
env_p   = partants["EnvironmentSatisfaction"].mean()
train_r = restants["TrainingTimesLastYear"].mean()
train_p = partants["TrainingTimesLastYear"].mean()

recommandations = [
    (
        "Revalorisation salariale ciblee",
        "MonthlyIncome",
        (f"Restants : {inc_r:.0f} roupies/mois | Partants : {inc_p:.0f} roupies/mois ({(inc_p-inc_r)/inc_r*100:+.1f}%)"
         "\nLe signal est modeste mais coherent avec la litterature RH : le salaire seul ne retient pas,"
         " mais une sous-remuneration perçue accelere la decision de partir."
         " -> Identifier les employes dont le salaire est inferieur a la mediane de leur"
         " tranche d'experience et les revaloriser avant qu'ils comparent avec le marche.")
    ),
    (
        "Priorite aux 3 premieres annees d'anciennete",
        "YearsAtCompany",
        (f"Restants : {years_r:.1f} ans d'anciennete | Partants : {years_p:.1f} ans ({(years_p-years_r)/years_r*100:+.1f}%)"
         "\nC'est le signal le plus fort apres les donnees de badgeage : les departs sont concentres"
         " dans les premiers annees. Les employes qui passent le cap des 5-6 ans partent beaucoup"
         " moins."
         " -> Renforcer l'onboarding, le suivi managarial la premiere annee, et proposer"
         " une premiere evolution de poste avant 3 ans.")
    ),
    (
        "Surveiller les heures supplementaires via les badges",
        "days_over_8h",
        (f"Restants : {over8_r:.0f} jours > 8h | Partants : {over8_p:.0f} jours ({(over8_p-over8_r)/over8_r*100:+.1f}%)"
         "\nC'est le signal le plus discriminant du jeu de donnees. Un employe qui fait"
         " systematiquement des heures supplementaires est souvent en situation de surcharge"
         " ou de desengagement silencieux avant depart."
         " -> Mettre en place un tableau de bord mensuel des heures par employe et"
         " declencher un entretien des qu'un employe depasse 3 semaines consecutives > 8h.")
    ),
    (
        "Rendre les enquetes de satisfaction actionnables",
        "JobSatisfaction + EnvironmentSatisfaction",
        (f"Satisfaction travail - Restants : {sat_r:.2f}/4 | Partants : {sat_p:.2f}/4 ({(sat_p-sat_r)/sat_r*100:+.1f}%)"
         f"\nSatisfaction environnement - Restants : {env_r:.2f}/4 | Partants : {env_p:.2f}/4 ({(env_p-env_r)/env_r*100:+.1f}%)"
         "\nLes deux scores sont plus bas chez les partants. L'enquete existe deja chez HumanForYou"
         " mais les resultats ne semblent pas declencher d'actions systematiques."
         " -> Definir un seuil d'alerte a 2.5/4 par departement et rendre obligatoire"
         " un plan d'action du manager dans les 30 jours si le seuil est franchi.")
    ),
    (
        "Maintenir un acces regulier a la formation",
        "TrainingTimesLastYear",
        (f"Restants : {train_r:.2f} formations/an | Partants : {train_p:.2f} formations/an"
         "\nLe signal est modeste mais coherent : la formation est un levier de fidelisation"
         " et un signal envoye a l'employe que l'entreprise investit en lui."
         " -> S'assurer qu'aucun employe ne passe une annee complete sans formation,"
         " en particulier dans les profils a risque identifies par le modele.")
    ),
    (
        "Systeme d'alerte precoce base sur le modele RF",
        "Score de probabilite de depart",
        (f"Le modele predit les departs avec AUC = {auc_opti:.3f} sur le jeu de test."
         f" Avec le seuil optimise, il capture la grande majorite des partants effectifs."
         "\n-> Calculer mensuellement le score de probabilite pour chaque employe."
         " Cibler les entretiens RH sur les profils dont la probabilite depasse 40%,"
         " avant que la decision soit prise.")
    ),
]

for i, (titre, source, desc) in enumerate(recommandations, 1):
    print("\n" + "="*65)
    print(f"Recommandation {i} : {titre}")
    print(f"Variable(s) : {source}")
    print("="*65)
    print(desc)


## 8. Limites du modele en production

Les performances mesurees sur le jeu de test sont solides, mais plusieurs points sont a connaitre avant tout deploiement.

### Signal badge non disponible en temps reel
`days_over_8h` est la variable la plus predictive (+96% chez les partants). Elle est calculee sur **une annee complete de pointage**. En production, ce signal n'est disponible qu'en fin d'annee ou apres 6 a 8 mois d'historique - il ne peut pas etre utilise pour des alertes mensuelles sans adaptation.

### AUC de 0.997 : interpreter avec prudence
Un AUC aussi eleve est rare. Il est en grande partie explique par `days_over_8h`. Sans cette variable, les performances seraient plus modestes. Ce chiffre ne doit pas etre presente comme representatif d'un modele deployable tel quel.

### Seuil choisi sur le jeu de test
Le seuil de decision optimise a ete selectionne en evaluant les resultats sur le jeu de test. En production, ce seuil devrait etre valide sur un jeu de donnees independant ou par validation croisee pour eviter un biais d'optimisme.

### Stabilite temporelle
Le modele est entraine sur des donnees 2015. Les conditions de travail, les politiques RH et le marche de l'emploi evoluent. Une mise a jour annuelle du modele est recommandee.

### Correlation, pas causalite
Le modele identifie des correlations, pas des causes. Augmenter le salaire d'un employe signale par le modele ne garantit pas qu'il reste - d'autres facteurs non mesures (projet personnel, opportunite externe) peuvent etre determinants.


## 9. Synthese du notebook

Ce notebook a repondu a la question centrale : **pourquoi les employes partent-ils, et comment le predire ?**

### Performances du modele final

| Metrique | RF de base | RF optimise |
|---|---|---|
| F1 (test) | calcule en cellule 3 | calcule en cellule 4 |
| AUC (test) | calcule en cellule 3 | calcule en cellule 4 |

### Top 5 des facteurs de depart (convergence Gini + Permutation)

1. **MonthlyIncome** - Les partants gagnent ~20% de moins que les restants
2. **YearsAtCompany** - Les departs surviennent surtout apres 1 a 3 ans
3. **YearsSinceLastPromotion** - Les partants attendent plus longtemps sans evolution
4. **days_over_8h / days_absent** - Signaux de desengagement detectables via les badges
5. **JobSatisfaction / EnvironmentSatisfaction** - Ecart ~0.3 point sur 4

### Ce qu'on peut faire concretement

- Identifier les profils a risque chaque mois avec le score de probabilite
- Agir sur le salaire et la carriere avant que la decision soit prise
- Utiliser l'absenteisme et les heures supplementaires comme signaux precoces
- Rendre les enquetes de satisfaction actionnables (seuil 2.5/4 = alerte)


## 10. Sauvegarde du modele final

In [ ]:
os.makedirs(MODELS, exist_ok=True)

joblib.dump(rf_opti, MODELS + "random_forest_final.pkl")
joblib.dump(list(X.columns), MODELS + "feature_columns.pkl")

print(f"Modele sauvegarde     -> {MODELS}random_forest_final.pkl")
print(f"Features sauvegardees -> {MODELS}feature_columns.pkl")
print(f"Nombre de features : {len(X.columns)}")
print(f"Parametres : n_estimators={rf_opti.n_estimators}, max_depth={rf_opti.max_depth}")
print(f"\nF1 final sur test  : {f1_opti:.4f}")
print(f"AUC final sur test : {auc_opti:.4f}")